In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==4.46.*",
        "accelerate==1.1.*"
    ],
    check=True
)

print("INSTALLATION COMPLETE")
print("RESTART SESSION NOW")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 88.9 MB/s eta 0:00:00
INSTALLATION COMPLETE
RESTART SESSION NOW


In [2]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
).to("cuda")

model.eval()


def prompt_of_len(context_length):
    token_id = tokenizer.encode(
        " benchmark",
        add_special_tokens=False
    )[0]

    return tokenizer.decode(
        [token_id] * context_length
    )


for context_length in (128, 512, 2048):
    prompt = prompt_of_len(context_length)

    token_count = len(
        tokenizer.encode(
            prompt,
            add_special_tokens=False
        )
    )

    print(
        "Requested:",
        context_length,
        "Actual:",
        token_count
    )

print("MODEL AND PROMPTS READY")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Requested: 128 Actual: 128
Requested: 512 Actual: 512
Requested: 2048 Actual: 2048
MODEL AND PROMPTS READY


In [3]:
cold_results = {}

for context_length in (128, 512, 2048):
    prompt = prompt_of_len(context_length)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()

    cold_results[context_length] = round(
        time.perf_counter() - start,
        4
    )

    print(
        context_length,
        cold_results[context_length],
        "seconds"
    )

print("BROKEN BENCHMARK COMPLETE")

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


128 2.2202 seconds
512 0.7752 seconds
2048 0.8584 seconds
BROKEN BENCHMARK COMPLETE


In [4]:
warmup_prompt = prompt_of_len(64)

warmup_inputs = tokenizer(
    warmup_prompt,
    return_tensors="pt"
).to("cuda")

with torch.inference_mode():
    _ = model.generate(
        **warmup_inputs,
        max_new_tokens=8,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

torch.cuda.synchronize()

print("WARM-UP COMPLETE")


fixed_results = {}

for context_length in (128, 512, 2048):
    prompt = prompt_of_len(context_length)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()

    fixed_results[context_length] = round(
        time.perf_counter() - start,
        4
    )

    print(
        context_length,
        fixed_results[context_length],
        "seconds"
    )

print("FIXED BENCHMARK COMPLETE")

WARM-UP COMPLETE
128 0.7642 seconds
512 0.7882 seconds
2048 0.8945 seconds
FIXED BENCHMARK COMPLETE


In [5]:
import json

REPORT_PATH = "/kaggle/working/warmup_benchmark_report.json"

report = {
    "model": MODEL_ID,
    "cold_no_warmup_seconds": cold_results,
    "fixed_after_warmup_seconds": fixed_results,
    "cold_start_confound_observed": (
        cold_results[128] > cold_results[512]
    ),
    "fixed_trend_is_monotonic": (
        fixed_results[128]
        < fixed_results[512]
        < fixed_results[2048]
    ),
    "diagnosis": (
        "The first 128-token measurement paid one-time CUDA and "
        "framework initialization costs, making the shortest prompt "
        "look slower than longer prompts."
    ),
    "fix": (
        "Run and discard a warm-up generation before timing the "
        "context-length sweep."
    )
}

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(report, file, indent=2)

assert cold_results[128] > cold_results[512]

assert (
    fixed_results[128]
    < fixed_results[512]
    < fixed_results[2048]
), (
    f"Expected latency to climb with context length, "
    f"got {fixed_results}"
)

print(json.dumps(report, indent=2))
print("GREEN CHECK: PASS")

{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "cold_no_warmup_seconds": {
    "128": 2.2202,
    "512": 0.7752,
    "2048": 0.8584
  },
  "fixed_after_warmup_seconds": {
    "128": 0.7642,
    "512": 0.7882,
    "2048": 0.8945
  },
  "cold_start_confound_observed": true,
  "fixed_trend_is_monotonic": true,
  "diagnosis": "The first 128-token measurement paid one-time CUDA and framework initialization costs, making the shortest prompt look slower than longer prompts.",
  "fix": "Run and discard a warm-up generation before timing the context-length sweep."
}
GREEN CHECK: PASS


In [6]:
import base64
from IPython.display import HTML, display

file_path = "/kaggle/working/warmup_benchmark_report.json"

with open(file_path, "rb") as file:
    encoded = base64.b64encode(file.read()).decode()

download_link = f"""
<a download="warmup_benchmark_report.json"
   href="data:application/json;base64,{encoded}">
   Download warmup_benchmark_report.json
</a>
"""

display(HTML(download_link))